# Marmousi01 — true velocity model: read & view

First step of the pre-refactor FWI-loop test: load and inspect the
Marmousi01 true velocity model — `float32`, shape 128×256 (a downsampled
Marmousi grid: 128 depth samples × 256 lateral samples).

The file lives in the sibling benchmark project
(`projetos/Benchmarks_FWI_ML/Dados/Marmousi01/marm_true.npy`); this
notebook stays in `ls-fwi` and only READS it.

Reference: the classic Marmousi1 model spans ~1500–5500 m/s
(water layer ~1500 m/s at the top).

In [ ]:
import numpy as np
from pathlib import Path

CANDIDATES = [
    Path("../../Benchmarks_FWI_ML/Dados/Marmousi01/marm_true.npy"),   # repo root
    Path("../../../Benchmarks_FWI_ML/Dados/Marmousi01/marm_true.npy"),  # notebooks/
    Path("/tgdesenv/src/proj0/ntbs/doutorado/5semestre.2024.2/estudo.orientado/"
         "projetos/Benchmarks_FWI_ML/Dados/Marmousi01/marm_true.npy"),  # host
    Path("/tgdesenv/doutorado/5semestre.2024.2/estudo.orientado/"
         "projetos/Benchmarks_FWI_ML/Dados/Marmousi01/marm_true.npy"),  # container
]
MARM = next(c for c in CANDIDATES if c.exists())
m = np.load(MARM)

print(f"file : {MARM}")
print(f"shape: {m.shape}  dtype: {m.dtype}")
print(f"range: {m.min():.1f} .. {m.max():.1f}")
print(f"mean : {m.mean():.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(m, cmap="turbo", aspect="auto", origin="upper")
ax.set_xlabel("lateral sample (of 256)")
ax.set_ylabel("depth sample (of 128)")
ax.set_title("Marmousi01 — true velocity model (row 0 = top of the section)")
fig.colorbar(im, ax=ax, label="velocity")
plt.tight_layout()
plt.show()

In [ ]:
# quick profiles: one velocity trace (vertical) and one horizontal
# layer profile — useful sanity checks before wiring an FWI loop
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 3.5))
axL.plot(m[:, m.shape[1] // 2], np.arange(m.shape[0]))
axL.invert_yaxis()
axL.set_xlabel("velocity")
axL.set_ylabel("depth sample")
axL.set_title(f"vertical trace @ lateral sample {m.shape[1] // 2}")
axR.plot(m[m.shape[0] // 2, :])
axR.set_xlabel("lateral sample")
axR.set_ylabel("velocity")
axR.set_title(f"horizontal profile @ depth sample {m.shape[0] // 2}")
plt.tight_layout()
plt.show()

## From Marmousi to the pipeline's model grid

The AE's generator works on **(672, 800)**: `Utils.find_size(depth=5)`
takes the raw (680, 800) models down to the largest sizes divisible by
2⁵ (the decoder's 5 upsampling levels), and the latent grid
(1, 512, 21, 25) is exactly 672/32 × 800/32.

The Marmousi file (128×256) is *smaller* in both dimensions, so this is
not a crop — it is a **resample** (bilinear upscale) to 672×800,
followed by the campaign's normalization.

**Normalization** (from `src/preprocessing/pipeline.py`, exactly as the
apps invoke it): `BatchAbsMaxScaler` is fitted on the TRAIN split's true
models — `absmax = max(|X|, axis=(0,1))` over the (900, 680, 800) stack,
i.e. one abs-max **per lateral column** — and every model (train and
test) is divided by it. The files are raw m/s; after scaling, velocities
become fractions of their column's train-set maximum (the context
figures' 0.4–0.9 colorbars). The Marmousi gets the same treatment:
resample FIRST (so the columns align), then divide by the same
per-column absmax.

In [ ]:
# resample 128x256 -> (672, 800), still in m/s
import torch

NZ, NX = 672, 800                     # nz_cut x nx_cut (find_size depth=5)
m_up = torch.nn.functional.interpolate(
    torch.from_numpy(m[None, None].astype(np.float32)),
    size=(NZ, NX), mode="bilinear", align_corners=False)[0, 0].numpy()
print(f"resampled: {m_up.shape}, range {m_up.min():.1f}..{m_up.max():.1f} m/s")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(m_up, cmap="turbo", aspect="auto")
ax.set_title("Marmousi resampled to the pipeline grid (m/s)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# the campaign's normalization: BatchAbsMaxScaler fitted on the TRAIN
# split (src/preprocessing/pipeline.py) — one abs-max per lateral column
from src.apps.global_vars import load_data_and_models

(data_basedir, dirs, train_data, test_data,
 train_models, test_models, train_indexes, test_indexes, newer) = \
    load_data_and_models(data='deep', data_type='deep', data_resolution='high',
                         data_max_samples=None, test_perc=0.1, seed=79)

raw_train = train_models.get_true_models().get_models()   # (900, 680, 800), raw m/s
absmax = np.max(np.abs(raw_train), axis=(0, 1))           # (800,) per lateral column
print(f"train true models: {raw_train.shape}, range "
      f"{raw_train.min():.1f}..{raw_train.max():.1f} m/s")
print(f"absmax per column: {absmax.shape}, "
      f"range {absmax.min():.1f}..{absmax.max():.1f} m/s")

m_ds = m_up / absmax                                      # broadcast over (672, 800)
print(f"normalized Marmousi: range {m_ds.min():.3f}..{m_ds.max():.3f} "
      "(context figures show ~0.4..0.9)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.imshow(m_up, cmap="turbo", aspect="auto"); ax1.set_title("resampled (m/s)")
ax2.imshow(m_ds, cmap="turbo", aspect="auto"); ax2.set_title("normalized (BatchAbsMaxScaler)")
plt.tight_layout()
plt.show()

## Invert: latent optimization against the Marmousi target

Reuses the campaign machinery exactly as `task_processing.py` does:
`EncoderDecoder` loaded from the refit checkpoint via skorch
(`load_params`), split into the Generator, then
`SkorchGeneratorInversionBuilder` with `latent_init='zeros'`
(no encoder needed — and per the reachability claim, no warm start
required), the campaign's best config (lr=0.1, 1000 epochs), and the
Marmousi target as `y`. `GeneratorInversion.forward(X)` ignores X —
the latent is the only trainable.

In [ ]:
import sys
import torch.nn as nn
from pathlib import Path

REPO = Path("..").resolve() if (Path("..") / "src").exists() else Path(".").resolve()
sys.path.insert(0, str(REPO))

from src.processing.ml_models import EncoderDecoder
from src.processing.ml_models_builder import (ScoredSkorchModel,
    SkorchGeneratorModelBuilder, SkorchGeneratorInversionBuilder)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# refit checkpoint of the best ML model (encdec 5x5, seed 79)
CKPT_DIR = Path("/gemini/tg/ntbs/doutorado/5semestre/estudo.orientado/"
                "deeptomograph_data/data_new_and_bigger/"
                "deeptomograph_results/predictions.encdec")
ckpts = sorted(CKPT_DIR.glob("REFIT__ENCDEC_ENCDEC_5x5*__params.pt"))
assert ckpts, f"no REFIT__ENCDEC_ENCDEC_5x5*__params.pt under {CKPT_DIR}"
print("checkpoints found:"); [print("  ", c.name) for c in ckpts]

# mirrors InferenceGrid's skorch_model_params (the campaign's wiring)
autoencoder = ScoredSkorchModel(
    module=EncoderDecoder, device=device, seed=79,
    max_epochs=1000, batch_size=16,
    criterion=nn.MSELoss(), optimizer=torch.optim.Adam, lr=1e-4,
    in_channels=1, out_channels=1, n_filters_init=16,
    pool_kernel_size=(2, 2), pool_stride=(2, 2),
    batch_normalization=True, dropout=0.2,
    attention_img_size=(21, 25), attention_in_channels=512,
    attention_hidden_size=512, attention_num_heads=16,
    attention_num_layers=12, attention_mlp_dim=4096, vis=False)
autoencoder.initialize()
autoencoder.load_params(f_params=str(ckpts[0]))

generator = SkorchGeneratorModelBuilder().build(
    autoencoder=autoencoder.module_, device=device)
generator.initialize()

# the campaign's forward logs latent.mean() every step — silence it here
from src.processing import ml_models as _mlm
_mlm.log.info = lambda *a, **k: None

inversion = SkorchGeneratorInversionBuilder().build(
    generator=generator.module_, latent_shape=(1, 512, 21, 25),
    latent_init="zeros", seed=79,
    max_epochs=1000, device=device, lr=0.1)   # best campaign config
inversion.initialize()

y = torch.from_numpy(m_ds[None, None].astype(np.float32)).to(device)
x = torch.zeros_like(y)                      # X is ignored by forward
inversion.partial_fit(x, y)
print("done — final train_loss:", inversion.history[-1]["train_loss"])

In [ ]:
# results — loss curve (with the pack's 95%/99% marks) and the recovery
losses = [h["train_loss"] for h in inversion.history]
final = losses[-1]
step95 = next(i for i, v in enumerate(losses) if v <= 1.05 * final)
step99 = next(i for i, v in enumerate(losses) if v <= 1.01 * final)
print(f"final MSE {final:.5f} | 95% at epoch {step95 + 1}, "
      f"99% at epoch {step99 + 1} (of {len(losses)})")

z = inversion.module_.latent.detach()
rec = generator.module_(z)[0, 0].detach().cpu().numpy()
tgt = y[0, 0].cpu().numpy()

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4))
axL.plot(losses)
axL.axvline(step95, ls="--", lw=1, alpha=0.6)
axL.axvline(step99, ls=":", lw=1, alpha=0.6)
axL.set_yscale("log")
axL.set_xlabel("epoch (1 backpropagation step each)")
axL.set_ylabel("inversion loss (model-space MSE)")
axL.set_title("convergence — zeros init, lr=0.1, 1000 epochs")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, img, ttl in zip(axes, [tgt, rec, tgt - rec],
                        ["target (Marmousi, dataset scale)",
                         "latent recovery (zeros init)",
                         "difference"]):
    ax.imshow(img, cmap="turbo", aspect="auto")
    ax.set_title(ttl, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()